In [ ]:
# Cell 1 — Install dependencies not available in the base Kaggle image
import subprocess

subprocess.run(["pip", "install", "timm", "librosa"], check=True)

In [ ]:
# Cell 2 — Imports and config
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import timm
import torch
import torch.nn as nn

# ---------------------------------------------------------------------------
# Audio / spectrogram constants — must match train.py exactly
# ---------------------------------------------------------------------------
SR = 32000
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 1024
FMIN = 20
FMAX = 16000
CLIP_DURATION = 5  # seconds per chunk
CLIP_SAMPLES = SR * CLIP_DURATION

# ImageNet normalisation applied after z-score (matches spec_to_tensor in train.py)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
DATA = Path("/kaggle/input/birdclef-2026")  # competition data
MODEL_DIR = Path("/kaggle/input/birdclef2026-efficientnet-b0")  # uploaded checkpoints

# ---------------------------------------------------------------------------
# Runtime config
# ---------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
STRIDE = 5.0  # non-overlapping 5s windows — aligns with submission row_id format

print(f"Device: {DEVICE}")
print(f"DATA:      {DATA}")
print(f"MODEL_DIR: {MODEL_DIR}")

In [ ]:
# Cell 3 — Model definition and audio utilities (copied from train.py)

# ---------------------------------------------------------------------------
# Model
# ---------------------------------------------------------------------------


class BirdModel(nn.Module):
    """EfficientNet (or any timm backbone) with a linear head for multilabel classification."""

    def __init__(self, backbone: str, n_classes: int, pretrained: bool = False):
        super().__init__()
        self.backbone = timm.create_model(
            backbone,
            pretrained=pretrained,
            num_classes=n_classes,
            in_chans=3,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)


# ---------------------------------------------------------------------------
# Audio utilities
# ---------------------------------------------------------------------------


def make_melspec(y: np.ndarray, sr: int = SR) -> np.ndarray:
    """Compute log-mel spectrogram from a 1-D audio array."""
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=N_MELS,
        hop_length=HOP_LENGTH,
        n_fft=N_FFT,
        fmin=FMIN,
        fmax=FMAX,
    )
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


def spec_to_tensor(spec: np.ndarray) -> torch.Tensor:
    """Z-score normalise + stack to 3-channel tensor for ImageNet-pretrained backbone."""
    # spec: (n_mels, time) in dB, roughly [-80, 0]
    spec = (spec - spec.mean()) / (spec.std() + 1e-6)
    img = np.stack([spec, spec, spec], axis=0)  # (3, n_mels, time)
    t = torch.from_numpy(img)
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t - mean) / std


print("Model class and audio utilities defined.")

In [ ]:
# Cell 4 — Load species list and model checkpoints

# Species list: sorted primary_label from taxonomy.csv (234 species).
# This must match the order used during training.
taxonomy = pd.read_csv(DATA / "taxonomy.csv")
species = sorted(taxonomy["primary_label"].astype(str).tolist())
n_species = len(species)
print(f"Species count: {n_species}")

# Find all checkpoint files in MODEL_DIR
ckpt_paths = sorted(MODEL_DIR.glob("*.pt"))
print(f"Checkpoints found: {len(ckpt_paths)}")
for p in ckpt_paths:
    print(f"  {p.name}")

# Load each checkpoint into a BirdModel and set to eval mode
models = []
for ckpt_path in ckpt_paths:
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    model = BirdModel(backbone="efficientnet_b0", n_classes=n_species).to(DEVICE)
    model.load_state_dict(ckpt["model"])
    model.eval()
    models.append(model)
    val_loss = ckpt.get("val_loss", float("nan"))
    epoch = ckpt.get("epoch", "?")
    print(f"  Loaded {ckpt_path.name} — epoch {epoch}, val_loss {val_loss:.4f}")

print(f"\nTotal models loaded: {len(models)}")

In [ ]:
# Cell 5 — Inference function


@torch.no_grad()
def predict_soundscape(
    path: Path,
    models: list,
    device: torch.device,
    stride: float = 5.0,
) -> np.ndarray:
    """Run sliding-window inference over a soundscape.

    Each window is CLIP_DURATION (5s) long. With stride=5.0 the windows are
    non-overlapping, which aligns with the submission format where each
    row_id covers one 5-second segment.

    Args:
        path:   Path to the soundscape audio file (.ogg or .wav).
        models: List of loaded BirdModel instances in eval mode.
        device: Torch device.
        stride: Step size in seconds between consecutive windows.

    Returns:
        preds: np.ndarray of shape (n_chunks, n_species) with sigmoid probabilities
               averaged across all models.
    """
    # Load full soundscape at target sample rate
    y, _ = librosa.load(path, sr=SR, mono=True)
    stride_samples = int(stride * SR)

    # Slice into fixed-length chunks
    chunks = []
    pos = 0
    while pos + CLIP_SAMPLES <= len(y):
        chunk = y[pos : pos + CLIP_SAMPLES]
        spec = make_melspec(chunk)
        chunks.append(spec_to_tensor(spec))
        pos += stride_samples

    # Edge case: soundscape shorter than one clip
    if not chunks:
        return np.zeros((1, n_species), dtype=np.float32)

    batch = torch.stack(chunks).to(device)  # (n_chunks, 3, n_mels, time)

    # Run each model and collect sigmoid predictions
    all_preds = []
    for model in models:
        logits = model(batch)  # (n_chunks, n_species)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())

    # Average sigmoid predictions across models
    preds = np.mean(all_preds, axis=0)  # (n_chunks, n_species)
    return preds


print("predict_soundscape() defined.")

In [ ]:
# Cell 6 — Run inference and build submission.csv

# Load sample submission to learn the expected columns and row_id format
sample_sub = pd.read_csv(DATA / "sample_submission.csv")
print(f"Sample submission shape: {sample_sub.shape}")
print(sample_sub.head(3))

# Find all test soundscapes (.ogg and .wav)
test_dir = DATA / "test_soundscapes"
soundscapes = sorted(test_dir.rglob("*.ogg")) + sorted(test_dir.rglob("*.wav"))
print(f"\nFound {len(soundscapes)} test soundscapes")

# Run inference soundscape by soundscape and build submission rows
rows = []
for sc_path in soundscapes:
    print(f"  Processing {sc_path.name} ...", end=" ", flush=True)
    preds = predict_soundscape(sc_path, models, DEVICE, stride=STRIDE)
    # row_id format: {soundscape_stem}_{end_second} at 5, 10, 15, ...
    for i, chunk_preds in enumerate(preds):
        end_sec = (i + 1) * 5
        row = {"row_id": f"{sc_path.stem}_{end_sec}"}
        for j, sp in enumerate(species):
            row[sp] = float(chunk_preds[j])
        rows.append(row)
    print(f"{len(preds)} chunks")

sub = pd.DataFrame(rows)

# Align columns to sample submission (keeps only species expected by the grader)
expected_cols = [c for c in sample_sub.columns if c != "row_id"]
present_cols = [c for c in expected_cols if c in sub.columns]
missing_cols = [c for c in expected_cols if c not in sub.columns]
if missing_cols:
    print(
        f"WARNING: {len(missing_cols)} expected columns missing from predictions, filling with 0"
    )
    for c in missing_cols:
        sub[c] = 0.0
sub = sub[["row_id"] + expected_cols]

# Save submission
out_path = Path("submission.csv")
sub.to_csv(out_path, index=False)
print(f"\nSubmission saved: {out_path}")
print(f"Shape: {sub.shape}")
print(sub.head(5))